In [ ]:
# -*- coding: utf-8 -*-
"""
PYCRAFT : AI Assistant
Enhanced with gradient background UI
"""

# !pip install -U transformers
# !pip install -q gradio transformers torch black autopep8 reportlab requests

import gradio as gr
from transformers import pipeline
import torch
import ast
import re
from datetime import datetime
from pathlib import Path
import black
import autopep8
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import requests

class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95

conversation_history = []
model_pipe = None

def load_model():
    global model_pipe
    print("🚀 Loading IBM Granite 3.3 2B Instruct model...")
    device = 0 if torch.cuda.is_available() else -1
    device_type = "GPU" if device == 0 else "CPU"
    print(f"   Using: {device_type}")
    model_pipe = pipeline(
        "text-generation",
        model=ChatbotConfig.MODEL_NAME,
        device=device,
        torch_dtype=torch.float16 if device == 0 else torch.float32,
    )
    print("✅ Model loaded successfully!")
    return model_pipe

def detect_python_code(text):
    code_patterns = [
        r'def\s+\w+\s*\(', r'class\s+\w+', r'import\s+\w+',
        r'from\s+\w+\s+import', r'if\s+.*:', r'for\s+.*\s+in\s+', r'while\s+.*:'
    ]
    return any(re.search(pattern, text) for pattern in code_patterns)

def check_syntax(code):
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Syntax Error (line {e.lineno}): {e.msg}"
    except Exception as e:
        return False, f"Error: {str(e)}"

def fix_code_indentation(code):
    lines = code.split('\n')
    return '\n'.join(line.rstrip().replace('\t', '    ') for line in lines)

def fix_code_brackets(code):
    for open_c, close_c in [('(', ')'), ('[', ']'), ('{', '}')]:
        diff = code.count(open_c) - code.count(close_c)
        if diff > 0:
            code += close_c * diff
    return code

def format_code(code):
    try:
        return black.format_str(code, mode=black.FileMode())
    except:
        try:
            return autopep8.fix_code(code)
        except:
            return code

def fix_python_code(code):
    code = fix_code_indentation(code)
    code = fix_code_brackets(code)
    is_valid, error_msg = check_syntax(code)
    if not is_valid:
        return {'success': False, 'fixed_code': code, 'error': error_msg,
                'message': f"⚠️ Could not fully fix code: {error_msg}"}
    code = format_code(code)
    return {'success': True, 'fixed_code': code, 'error': None,
            'message': "✅ Code fixed and formatted successfully!"}

def generate_response(user_message, temperature, max_tokens, top_p):
    conversation_history.append({"role": "user", "content": user_message})
    try:
        response = model_pipe(
            conversation_history,
            max_new_tokens=int(max_tokens),
            temperature=temperature,
            do_sample=True,
            top_p=top_p
        )
        assistant_message = response[0]['generated_text']
        if isinstance(assistant_message, list):
            assistant_message = assistant_message[-1].get("content", str(assistant_message))
        conversation_history.append({"role": "assistant", "content": assistant_message})
        return assistant_message
    except Exception as e:
        error_response = f"❌ Error: {str(e)}"
        conversation_history.append({"role": "assistant", "content": error_response})
        return error_response

def process_user_input(user_message, temperature, max_tokens, top_p, chat_history):
    if not user_message.strip():
        return chat_history
    if detect_python_code(user_message):
        code_pattern = r'```python\n(.*?)\n```|```\n(.*?)\n```'
        code_match = re.search(code_pattern, user_message, re.DOTALL)
        if code_match:
            code_snippet = code_match.group(1) or code_match.group(2)
            fix_result = fix_python_code(code_snippet)
            if fix_result['success']:
                response_text = f"{fix_result['message']}\n\n**Fixed Code:**\n```python\n{fix_result['fixed_code']}\n```"
            else:
                response_text = (f"{fix_result['message']}\n\n**Original Code:**\n```python\n{code_snippet}\n```\n\n"
                                 f"**Attempted Fix:**\n```python\n{fix_result['fixed_code']}\n```")
            chat_history.append((user_message, response_text))
            ai_response = generate_response(
                f"I fixed this Python code:\n```python\n{fix_result['fixed_code']}\n```\nBriefly explain what it does.",
                temperature, max_tokens, top_p
            )
            chat_history.append(("Analysis", ai_response))
            return chat_history
    ai_response = generate_response(user_message, temperature, max_tokens, top_p)
    chat_history.append((user_message, ai_response))
    return chat_history

def clear_chat():
    global conversation_history
    conversation_history = []
    return []

def download_txt():
    if not conversation_history:
        return "❌ Chat is empty!", None
    content = f"PYCRAFT : AI Assistant - Chat History\nExported: {datetime.now()}\n{'='*80}\n\n"
    for i, msg in enumerate(conversation_history, 1):
        content += f"\n[Message {i}]\n{msg['role'].upper()}:\n{'-'*80}\n{msg['content']}\n{'-'*80}\n"
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"pycraft_chat_{timestamp}.txt"
    try:
        Path(filename).write_text(content)
        return f"✅ Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None

def download_pdf():
    if not conversation_history:
        return "❌ Chat is empty!", None
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"pycraft_chat_{timestamp}.pdf"
    try:
        doc = SimpleDocTemplate(filename, pagesize=A4, topMargin=0.5*inch, bottomMargin=0.5*inch)
        styles = getSampleStyleSheet()
        title_style = ParagraphStyle('Title', parent=styles['Heading1'], fontSize=20,
                                     spaceAfter=12, alignment=1, fontName='Helvetica-Bold')
        user_style = ParagraphStyle('User', parent=styles['Normal'], fontSize=10,
                                    spaceAfter=8, spaceBefore=8, leftIndent=20, rightIndent=20,
                                    backColor='#E6F0FF', borderColor='#0033CC', borderWidth=1, borderPadding=8)
        bot_style  = ParagraphStyle('Bot',  parent=styles['Normal'], fontSize=10,
                                    spaceAfter=8, spaceBefore=8, leftIndent=20, rightIndent=20,
                                    backColor='#F3E5F5', borderColor='#7b1fa2', borderWidth=1, borderPadding=8)
        story = [
            Paragraph("⚡ PYCRAFT : AI Assistant - Chat History", title_style),
            Paragraph(f"<b>Exported:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']),
            Paragraph(f"<b>Total Messages:</b> {len(conversation_history)}", styles['Normal']),
            Spacer(1, 0.3*inch),
        ]
        for i, msg in enumerate(conversation_history, 1):
            clean = msg['content'].replace('<', '&lt;').replace('>', '&gt;')
            story.append(Paragraph(f"<b>Message #{i} — {msg['role'].upper()}</b>", styles['Normal']))
            story.append(Paragraph(clean, user_style if msg['role'] == 'user' else bot_style))
            story.append(Spacer(1, 0.2*inch))
            if i % 3 == 0 and i != len(conversation_history):
                story.append(PageBreak())
        story += [Spacer(1, 0.3*inch),
                  Paragraph("<b>End of Chat History</b> | Generated by PYCRAFT AI", styles['Normal'])]
        doc.build(story)
        return f"✅ Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None

def generate_share_link():
    if not conversation_history:
        return "❌ Chat is empty!"
    try:
        chat_content = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in conversation_history)
        resp = requests.post(
            "https://pastebin.com/api/api_post.php",
            data={'api_dev_key': '', 'api_option': 'paste', 'api_paste_code': chat_content,
                  'api_paste_name': f'PYCRAFT Chat {datetime.now().strftime("%Y-%m-%d %H:%M")}',
                  'api_paste_expire_date': 'N', 'api_paste_private': '0'},
            timeout=5
        )
        if 'pastebin.com' in resp.text:
            return f"✅ SHAREABLE LINK!\n\n🔗 {resp.text.strip()}\n\nShare anywhere!"
    except:
        pass
    return (f"✅ SHAREABLE SUMMARY:\n\nMessages: {len(conversation_history)}\n"
            f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\nDownload TXT/PDF to share!")


# ─────────────────────────────────────────────────────────────────
#  CSS — Pure dark theme with visible title
# ─────────────────────────────────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@700;900&family=Share+Tech+Mono&family=Inter:wght@400;500;600&display=swap');

/* ── Dark base ── */
body,
.gradio-container,
.gradio-container > .main,
#root,
.app {
    background-color: #0d0d0d !important;
    background-image: none !important;
    color: #e2e2e2 !important;
    font-family: 'Inter', sans-serif !important;
}

/* ── Panels & cards ── */
.block, .gr-block, .gr-form, .gr-box,
.gradio-row, .gradio-column,
.gr-panel, .panel, .gap,
.contain, .wrap, .tabitem {
    background-color: #141414 !important;
    border: 1px solid #2a2a2a !important;
    border-radius: 12px !important;
    box-shadow: 0 4px 20px rgba(0,0,0,0.6) !important;
}

/* ── Chatbot window ── */
.chatbot, .gr-chatbot,
[data-testid="chatbot"],
#chatbot {
    background-color: #111 !important;
    border: 1px solid #1e3a5f !important;
    border-radius: 14px !important;
}

/* User message bubble */
[data-testid="chatbot"] .message.user,
.gr-chatbot .message.user {
    background-color: #0f2744 !important;
    border: 1px solid #1a4a80 !important;
    border-radius: 16px 16px 4px 16px !important;
    color: #a8d4ff !important;
}

/* Bot message bubble */
[data-testid="chatbot"] .message.bot,
.gr-chatbot .message.bot {
    background-color: #1a0d2e !important;
    border: 1px solid #3d1f6e !important;
    border-radius: 16px 16px 16px 4px !important;
    color: #d4b8ff !important;
}

/* ── Text inputs ── */
textarea,
input[type="text"],
input[type="search"],
.gr-textbox textarea {
    background-color: #1a1a1a !important;
    border: 1px solid #333 !important;
    border-radius: 10px !important;
    color: #e2e2e2 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 0.95rem !important;
    transition: border-color 0.25s, box-shadow 0.25s !important;
    caret-color: #4fc3f7 !important;
}
textarea:focus,
input[type="text"]:focus {
    border-color: #4fc3f7 !important;
    box-shadow: 0 0 0 3px rgba(79,195,247,0.15) !important;
    outline: none !important;
}
textarea::placeholder { color: #555 !important; }

/* ── Sliders ── */
input[type="range"] {
    accent-color: #4fc3f7 !important;
    background: transparent !important;
}

/* ── Labels ── */
label, .gr-input-label, span.label-wrap, .label-wrap span {
    color: #888 !important;
    font-size: 0.78rem !important;
    font-weight: 500 !important;
    letter-spacing: 0.8px !important;
    text-transform: uppercase !important;
    font-family: 'Inter', sans-serif !important;
}

/* ── SEND / primary button ── */
button.primary, .gr-button-primary,
button[variant="primary"] {
    background: linear-gradient(135deg, #0077b6, #023e8a) !important;
    border: none !important;
    border-radius: 10px !important;
    color: #ffffff !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    box-shadow: 0 3px 12px rgba(0,119,182,0.4) !important;
    transition: all 0.25s ease !important;
    font-family: 'Inter', sans-serif !important;
}
button.primary:hover, .gr-button-primary:hover {
    background: linear-gradient(135deg, #0096c7, #0077b6) !important;
    box-shadow: 0 6px 20px rgba(0,150,199,0.5) !important;
    transform: translateY(-1px) !important;
}

/* ── CLEAR / stop button ── */
button.stop, .gr-button-stop,
button[variant="stop"] {
    background: linear-gradient(135deg, #c62828, #7f0000) !important;
    border: none !important;
    border-radius: 10px !important;
    color: #ffffff !important;
    font-weight: 600 !important;
    transition: all 0.25s ease !important;
    font-family: 'Inter', sans-serif !important;
}
button.stop:hover {
    background: linear-gradient(135deg, #ef5350, #c62828) !important;
    box-shadow: 0 6px 20px rgba(198,40,40,0.45) !important;
    transform: translateY(-1px) !important;
}

/* ── Secondary button ── */
button.secondary, .gr-button-secondary,
button[variant="secondary"] {
    background: #1e1e1e !important;
    border: 1px solid #3d1f6e !important;
    border-radius: 10px !important;
    color: #b39ddb !important;
    font-weight: 600 !important;
    transition: all 0.25s ease !important;
    font-family: 'Inter', sans-serif !important;
}
button.secondary:hover {
    background: #2a1a4a !important;
    border-color: #7c4dff !important;
    box-shadow: 0 4px 14px rgba(124,77,255,0.3) !important;
    transform: translateY(-1px) !important;
}

/* ── Markdown headings in sidebar ── */
.gr-markdown h3, .prose h3 {
    color: #4fc3f7 !important;
    font-size: 0.78rem !important;
    font-weight: 700 !important;
    letter-spacing: 2px !important;
    text-transform: uppercase !important;
    font-family: 'Inter', sans-serif !important;
    border-bottom: 1px solid #1e3a5f !important;
    padding-bottom: 4px !important;
    margin-top: 12px !important;
}
.gr-markdown p, .gr-markdown li, .prose p {
    color: #999 !important;
    font-size: 0.85rem !important;
}
.gr-markdown hr, .prose hr {
    border-color: #222 !important;
}

/* ── File upload box ── */
.gr-file, .file-preview {
    background-color: #161616 !important;
    border: 1px dashed #333 !important;
    border-radius: 10px !important;
    color: #888 !important;
}

/* ── Status textbox ── */
.gr-textbox {
    background-color: #141414 !important;
    border: 1px solid #2a2a2a !important;
    border-radius: 10px !important;
    color: #e2e2e2 !important;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: #111; }
::-webkit-scrollbar-thumb { background: #333; border-radius: 4px; }
::-webkit-scrollbar-thumb:hover { background: #4fc3f7; }

/* ── Selection ── */
::selection { background: rgba(79,195,247,0.25); color: #fff; }
"""

# ─────────────────────────────────────────────────────────────────
load_model()

with gr.Blocks(
    title="PYCRAFT : AI Assistant",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.blue,
        secondary_hue=gr.themes.colors.purple,
        neutral_hue=gr.themes.colors.gray,
    ),
    css=custom_css,
) as demo:

    # ── Header — fully inline styled for guaranteed visibility ──
    gr.HTML("""
    <link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@900&family=Share+Tech+Mono&display=swap" rel="stylesheet">
    <div style="background:#0d0d0d;padding:32px 20px 20px;text-align:center;border-bottom:1px solid #1e3a5f;margin-bottom:8px;">
        <div style="font-family:'Orbitron','Courier New',monospace;font-size:2.6rem;font-weight:900;color:#4fc3f7;letter-spacing:8px;text-transform:uppercase;text-shadow:0 0 10px rgba(79,195,247,0.9),0 0 30px rgba(79,195,247,0.5),0 0 60px rgba(79,195,247,0.2);margin-bottom:8px;">
            ⚡ PYCRAFT
        </div>
        <div style="font-family:'Share Tech Mono','Courier New',monospace;font-size:0.75rem;color:#546e7a;letter-spacing:5px;text-transform:uppercase;margin-bottom:18px;">
            AI ASSISTANT &nbsp;·&nbsp; PYTHON CODE FIXER &nbsp;·&nbsp; NO API KEYS
        </div>
        <div style="height:1px;background:linear-gradient(90deg,transparent 0%,#1e3a5f 20%,#4fc3f7 50%,#1e3a5f 80%,transparent 100%);max-width:600px;margin:0 auto;"></div>
    </div>
    """)

    with gr.Row():
        # Main chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="", height=430, show_copy_button=True, bubble_full_width=False)
            with gr.Row():
                msg = gr.Textbox(
                    label="",
                    placeholder="💬  Ask anything or paste Python code to auto-fix...",
                    lines=2, scale=4,
                )
                submit_btn = gr.Button("SEND ➤", scale=1, size="lg", variant="primary")
            with gr.Row():
                temperature = gr.Slider(label="🌡️ Temperature", minimum=0.1, maximum=1.0, value=0.7, step=0.05)
                max_tokens  = gr.Slider(label="📝 Max Tokens",  minimum=50,  maximum=1024, value=256, step=50)
                top_p       = gr.Slider(label="🎯 Top-P",       minimum=0.1, maximum=1.0, value=0.95, step=0.05)
            submit_btn.click(process_user_input, inputs=[msg, temperature, max_tokens, top_p, chatbot], outputs=[chatbot])
            msg.submit(process_user_input,        inputs=[msg, temperature, max_tokens, top_p, chatbot], outputs=[chatbot])

        # Sidebar
        with gr.Column(scale=1):
            gr.Markdown("### 🎮 Controls")
            clear_btn = gr.Button("🗑️ Clear Chat", variant="stop", size="sm")
            clear_btn.click(clear_chat, outputs=[chatbot])

            gr.Markdown("---")
            gr.Markdown("### 📥 Export")
            txt_btn  = gr.Button("📄 Download TXT", size="sm", variant="primary")
            pdf_btn  = gr.Button("📋 Download PDF", size="sm", variant="primary")
            txt_file = gr.File(label="TXT File")
            pdf_file = gr.File(label="PDF File")
            status   = gr.Textbox(label="Status", interactive=False, lines=2)
            txt_btn.click(download_txt, outputs=[status, txt_file])
            pdf_btn.click(download_pdf, outputs=[status, pdf_file])

            gr.Markdown("---")
            gr.Markdown("### 🔗 Share")
            share_btn    = gr.Button("🚀 Generate Share Link", size="sm", variant="secondary")
            share_output = gr.Textbox(label="Share Link", interactive=False, lines=5)
            share_btn.click(generate_share_link, outputs=[share_output])

            gr.Markdown("---")
            gr.Markdown("""
### ℹ️ About
**PYCRAFT AI** — IBM Granite 3.3-2B

- 🐍 Auto-detect & fix Python
- 💬 Full conversation context
- 📁 Export TXT / PDF
- 🔗 Share sessions
            """)

print("\n PYCRAFT AI ASSISTANT LAUNCHED!\n")
demo.launch(share=True, show_error=True, show_api=False)


🚀 Loading IBM Granite 3.3 2B Instruct model...
   Using: CPU


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

✅ Model loaded successfully!


/tmp/ipykernel_9716/3902112344.py:413: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_9716/3902112344.py:413: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_9716/3902112344.py:440: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="", height=430, show_copy_button=True, bubble_full_width=False)
/tmp/ipykernel_9716/3902112344.py:440: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to


 PYCRAFT AI ASSISTANT LAUNCHED!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://804200d18dcfeb779f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
